<a href="https://colab.research.google.com/github/pinkprincess536/eeg/blob/main/eeg.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install mne numpy pandas matplotlib scipy scikit-learn torch

In [ ]:
import mne
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from scipy.signal import spectrogram
import os, re, subprocess

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 1: Config Panel

**All tunable parameters in one place.**

In [ ]:
BASE_DIR = "/content/drive/MyDrive/EEG_PROJECT"
os.makedirs(BASE_DIR, exist_ok=True)

# --- ALL 24 CHB-MIT PATIENTS ---
ALL_PATIENTS = [f"chb{i:02d}" for i in range(1, 25)]

# Deterministic 75/25 train/test split (seed=42)
np.random.seed(42)
shuffled = ALL_PATIENTS.copy()
np.random.shuffle(shuffled)
split_idx = int(len(shuffled) * 0.75)
TRAIN_PATIENTS = sorted(shuffled[:split_idx])
TEST_PATIENTS  = sorted(shuffled[split_idx:])

print(f"All patients: {len(ALL_PATIENTS)}")
print(f"Train ({len(TRAIN_PATIENTS)}): {TRAIN_PATIENTS}")
print(f"Test  ({len(TEST_PATIENTS)}): {TEST_PATIENTS}")

# --- PIPELINE PARAMETERS ---
WINDOW_SIZE_SEC = 7.0
OVERLAP         = 0.3
LOWCUT          = 0.5
HIGHCUT         = 40.0
NOTCH_FREQ      = 60.0
CHANNEL_IDX     = 0

# --- TRAINING ---
BATCH_SIZE           = 8
LEARNING_RATE        = 0.001
NUM_EPOCHS           = 10
RANDOM_SEED          = 42
NON_SEIZURE_SAMPLES  = 500   # increased for full dataset

print(f"Window: {WINDOW_SIZE_SEC}s  |  Overlap: {OVERLAP}  |  Filter: {LOWCUT}-{HIGHCUT} Hz")
print(f"Non-seizure samples: {NON_SEIZURE_SAMPLES}")

## Cell 2: Download CHB-MIT Data

In [ ]:
# 6 PARALLEL DOWNLOADS -- ~4x faster than sequential
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

start_time = time.time()
total_bytes = 0

for i in range(1, 25):
    pid = f"chb{i:02d}"
    pdir = os.path.join(BASE_DIR, pid)
    os.makedirs(pdir, exist_ok=True)

    # 1. Download summary FIRST
    summary_url = f"https://physionet.org/files/chbmit/1.0.0/{pid}/{pid}-summary.txt"
    summary_path = os.path.join(BASE_DIR, f"{pid}-summary.txt")
    if not os.path.exists(summary_path):
        subprocess.run(["wget", "-q", "-O", summary_path, summary_url])
    total_bytes += os.path.getsize(summary_path)

    # 2. Try RECORDS file
    records_url = f"https://physionet.org/files/chbmit/1.0.0/{pid}/RECORDS"
    records_path = os.path.join(pdir, "RECORDS")
    subprocess.run(["wget", "-q", "-O", records_path, records_url])

    edf_list = []
    try:
        with open(records_path, "r") as f:
            edf_list = [l.strip() for l in f.read().splitlines() if l.strip().endswith(".edf")]
    except:
        pass

    source = "RECORDS"
    if len(edf_list) < 5:
        try:
            with open(summary_path, "r") as f:
                content = f.read()
            edf_list = re.findall(r"File Name:\s*(\S+\.edf)", content)
            source = "summary"
        except:
            pass

    n_files = len(edf_list)
    base_url = f"https://physionet.org/files/chbmit/1.0.0/{pid}/"

    # 3. Build download task list
    tasks = []
    for fname in edf_list:
        fname = fname.strip()
        if not fname:
            continue
        tasks.append((fname, os.path.join(pdir, fname), base_url))

    print(f"\n{pid}: {len(tasks)} files ({source}) [6 parallel]")

    # 4. Download 6 files simultaneously
    def download_one(args):
        fname, edf_path, url_base = args
        downloaded = False
        if not os.path.exists(edf_path):
            subprocess.run(["wget", "-q", "-O", edf_path, url_base + fname],
                           stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            downloaded = True
        size_mb = os.path.getsize(edf_path) / (1024*1024)
        # Download .seizures annotation file
        sez_path = edf_path + ".seizures"
        if not os.path.exists(sez_path):
            subprocess.run(["wget", "-q", "-O", sez_path, url_base + fname + ".seizures"],
                           stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        return fname, size_mb, downloaded

    done = 0
    new_count = 0
    with ThreadPoolExecutor(max_workers=6) as pool:
        futures = [pool.submit(download_one, t) for t in tasks]
        for future in as_completed(futures):
            fname, size_mb, is_new = future.result()
            done += 1
            marker = " NEW" if is_new else ""
            if is_new:
                new_count += 1
            print(f"  [{done}/{len(tasks)}] {fname}  ({size_mb:.0f} MB){marker}")

    elapsed = time.time() - start_time
    total_bytes += sum(os.path.getsize(os.path.join(pdir, f)) for f in edf_list)
    print(f"  -> {new_count} new, {done-new_count} existed | Total: {total_bytes/1e9:.1f} GB | {elapsed/60:.0f} min")

print(f"\n===== DONE =====")
elapsed = time.time() - start_time
print(f"Total: {total_bytes/1e9:.2f} GB in {elapsed/60:.1f} min ({total_bytes/(1e6*max(1,elapsed)):.1f} MB/s)")

## Cell 3: parse_seizure_summary()

Reads summary.txt -> lookup table. No more hardcoded timestamps.

In [ ]:
def parse_seizure_summary(summary_path):
    with open(summary_path, "r") as f:
        content = f.read()
    sections = re.split(r"File Name:\s*", content)[1:]
    sm = {}
    for s in sections:
        m = re.match(r"(\S+)", s)
        if not m:
            continue
        fn = m.group(1)
        st = [int(x) for x in re.findall(r"Seizure\s+Start\s+Time:\s*(\d+)\s*seconds", s)]
        en = [int(x) for x in re.findall(r"Seizure\s+End\s+Time:\s*(\d+)\s*seconds", s)]
        sm[fn] = list(zip(st, en))
    return sm

# Build seizure lookup for ALL patients
SEIZURE_MAP = {}
for p in ALL_PATIENTS:
    summary_path = os.path.join(BASE_DIR, f"{p}-summary.txt")
    if os.path.exists(summary_path):
        SEIZURE_MAP[p] = parse_seizure_summary(summary_path)
        n_seiz_files = sum(1 for v in SEIZURE_MAP[p].values() if v)
        print(f"{p}: {len(SEIZURE_MAP[p])} recordings, {n_seiz_files} with seizures")
    else:
        SEIZURE_MAP[p] = {}
        print(f"{p}: no summary, skipping")

## Cell 4: bandpass_filter()

Filter full recording ONCE before windowing. No per-window filtering.

In [ ]:
def bandpass_filter(raw, lowcut=0.5, highcut=40.0, notch_freq=60.0):
    rf = raw.copy()
    rf.filter(lowcut, highcut, fir_design="firwin", verbose=False)
    rf.notch_filter(freqs=notch_freq, verbose=False)
    return rf.get_data()

## Cell 5: create_windows()

30% overlap: stride=1254 samples (7s@256Hz). Was 50% -> 896.

In [ ]:
def create_windows(signal, sfreq, window_size_sec=7.0, overlap=0.3):
    ws = int(window_size_sec * sfreq)
    st = int(ws * (1 - overlap))
    total = signal.shape[1]
    wins = []
    for s in range(0, total - ws, st):
        wins.append(signal[:, s:s+ws])
    return np.array(wins)

## Cell 6: create_labels()

Soft overlap: any part of window touches seizure -> label=1.

In [ ]:
def create_labels(windows, sfreq, seizure_intervals):
    n = windows.shape[0]
    ws = windows.shape[2]
    stride = int(ws * (1 - 0.3))
    labs = np.zeros(n, dtype=int)
    if not seizure_intervals: return labs
    for i in range(n):
        a = (i*stride)/sfreq
        b = (i*stride+ws)/sfreq
        for st,en in seizure_intervals:
            if b >= st and a <= en:
                labs[i] = 1; break
    return labs

## Cell 7: create_spectrograms()

No filtering inside. Signal already filtered.

In [ ]:
def create_spectrograms(windows, sfreq, channel_idx=0):
    specs = []
    for win in windows:
        sig = win[channel_idx]
        _,_,S = spectrogram(sig, fs=sfreq, nperseg=256, noverlap=128)
        specs.append(S)
    return np.array(specs)

## Cell 8: process_recording()

Orchestrator. `return_windows=True` skips spectrogram for 1D CNN.

In [ ]:
def process_recording(edf_path, seizure_intervals,
                      window_size_sec=7.0, overlap=0.3,
                      lowcut=0.5, highcut=40.0, notch_freq=60.0,
                      channel_idx=0,
                      return_windows=False):
    """Load, filter, window, label. Optionally return raw windows."""
    raw = mne.io.read_raw_edf(edf_path, preload=True, verbose=False)
    signal = bandpass_filter(raw, lowcut, highcut, notch_freq)
    sfreq = raw.info["sfreq"]
    windows = create_windows(signal, sfreq, window_size_sec, overlap)
    labels = create_labels(windows, sfreq, seizure_intervals)
    if return_windows:
        return windows, labels
    else:
        specs = create_spectrograms(windows, sfreq, channel_idx)
        return specs, labels

## Cell 9: Process Training Set (chb01 + chb02)

Loops over all recordings, calls process_recording(), concatenates.

In [ ]:
def collect_recordings(patient_ids, seizure_map, base_dir,
                        window_size_sec=7.0, overlap=0.3,
                        return_windows=False):
    """Process all recordings -> aggregate dataset."""
    all_data, all_labels = [], []
    for pid in patient_ids:
        pdir = os.path.join(base_dir, pid)
        edf_files = sorted([f for f in os.listdir(pdir) if f.endswith(".edf")])
        for edf in edf_files:
            intervals = seizure_map[pid].get(edf, [])
            path = os.path.join(pdir, edf)
            data, labels = process_recording(path, intervals,
                window_size_sec=window_size_sec, overlap=overlap,
                return_windows=return_windows)
            all_data.append(data)
            all_labels.append(labels)
            n = np.sum(labels)
            print(f"  {pid}/{edf} -> {data.shape[0]} windows, {n} seizure")
    return (np.concatenate(all_data, axis=0), np.concatenate(all_labels, axis=0))

print("\n=== PROCESSING TRAIN SET (chb01 + chb02) ===")
X_train_full, y_train_full = collect_recordings(
    TRAIN_PATIENTS, SEIZURE_MAP, BASE_DIR,
    window_size_sec=WINDOW_SIZE_SEC, overlap=OVERLAP)

print(f"\nTrain: X={X_train_full.shape}, y={y_train_full.shape}")
print(f"Seizure: {np.sum(y_train_full)}, Non-seizure: {len(y_train_full)-np.sum(y_train_full)}")

## Cell 10: Process Test Set (chb03 ONLY)

Split by PATIENT, not random. True generalization test.

In [ ]:
print("\n=== PROCESSING TEST SET (chb03) ===")
X_test_full, y_test_full = collect_recordings(
    TEST_PATIENTS, SEIZURE_MAP, BASE_DIR,
    window_size_sec=WINDOW_SIZE_SEC, overlap=OVERLAP)

print(f"\nTest: X={X_test_full.shape}, y={y_test_full.shape}")
print(f"Seizure: {np.sum(y_test_full)}, Non-seizure: {len(y_test_full)-np.sum(y_test_full)}")

## Cell 11: Handle Class Imbalance

In [ ]:
def balance_dataset(specs, labels, n_non=200, seed=42):
    np.random.seed(seed)
    si = np.where(labels==1)[0]
    ni = np.where(labels==0)[0]
    cn = np.random.choice(ni, size=min(n_non,len(ni)), replace=False)
    k = np.concatenate([si, cn])
    np.random.shuffle(k)
    return specs[k], labels[k]

X_train, y_train = balance_dataset(X_train_full, y_train_full, NON_SEIZURE_SAMPLES, RANDOM_SEED)
X_test, y_test   = balance_dataset(X_test_full,  y_test_full,  NON_SEIZURE_SAMPLES, RANDOM_SEED)
ut,ct=np.unique(y_train,return_counts=True); print(f"Train balanced: {X_train.shape}, {dict(zip(ut,ct))}")
ut,ct=np.unique(y_test,return_counts=True);  print(f"Test balanced:  {X_test.shape}, {dict(zip(ut,ct))}")

## Cell 12: Normalize (Z-score)

In [ ]:
X_train = X_train.astype(np.float32); X_test = X_test.astype(np.float32)
tm = X_train.mean(); ts = X_train.std()
X_train = (X_train-tm)/ts; X_test = (X_test-tm)/ts
print(f"Normalized. mean={X_train.mean():.3f}, std={X_train.std():.3f}")

## Cell 13: Reshape for 2D CNN Input

In [ ]:
X_train = np.expand_dims(X_train, axis=1); X_test = np.expand_dims(X_test, axis=1)
print(f"CNN input: {X_train.shape}  (batch, channel, freq, time)")

## Cell 14: PyTorch Tensors

In [ ]:
Xtr = torch.tensor(X_train, dtype=torch.float32); ytr = torch.tensor(y_train, dtype=torch.long)
Xte = torch.tensor(X_test, dtype=torch.float32); yte = torch.tensor(y_test, dtype=torch.long)
tds = TensorDataset(Xtr,ytr); eds = TensorDataset(Xte,yte)
tl = DataLoader(tds, batch_size=BATCH_SIZE, shuffle=True)
el = DataLoader(eds, batch_size=BATCH_SIZE, shuffle=False)
print(f"Train batches: {len(tl)} | Test: {len(el)}")

## Cell 15: 2D CNN (Reference)

AdaptiveAvgPool2d((4,4)) for robust input sizes.

In [ ]:
class EEGCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1,16,kernel_size=3,padding=1)
        self.pool = nn.MaxPool2d(2)
        self.adapt = nn.AdaptiveAvgPool2d((4,4))
        self.fc1 = nn.Linear(16*4*4,32)
        self.fc2 = nn.Linear(32,2)
    def forward(self,x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.adapt(x)
        x = torch.flatten(x,1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = EEGCNN()
d = torch.randn(1,1,X_train.shape[2],X_train.shape[3])
with torch.no_grad(): o = model(d)
print(f"Model: {sum(p.numel() for p in model.parameters()):,} params")
print(f"Input {tuple(d.shape)} -> Output {tuple(o.shape)}  OK!")

## Cell 16: Train 2D CNN

In [ ]:
crit = nn.CrossEntropyLoss()
opt = optim.Adam(model.parameters(), lr=LEARNING_RATE)
for ep in range(NUM_EPOCHS):
    model.train(); rl = 0.0
    for x,y in tl:
        opt.zero_grad()
        out = model(x); loss = crit(out,y)
        loss.backward(); opt.step()
        rl += loss.item()
    print(f"Epoch {ep+1}/{NUM_EPOCHS}, Loss: {rl/len(tl):.4f}")

## Cell 17: Evaluate 2D CNN on chb03

In [ ]:
model.eval(); c,t=0,0; ap,at=[],[]
with torch.no_grad():
    for x,y in el:
        o=model(x); _,p=torch.max(o,1)
        ap.extend(p.cpu().numpy()); at.extend(y.cpu().numpy())
        t+=y.size(0); c+=(p==y).sum().item()
ap=np.array(ap); at=np.array(at)
print(f"Accuracy: {100*c/t:.2f}% ({c}/{t})")
sm=at==1
if sm.sum()>0:
    tp=(ap[sm]==1).sum(); fn=(ap[sm]==0).sum()
    print(f"Sensitivity: {tp/(tp+fn):.2f} ({tp}/{tp+fn})")
nm=at==0
if nm.sum()>0:
    tn=(ap[nm]==0).sum(); fp=(ap[nm]==1).sum()
    print(f"Specificity: {tn/(tn+fp):.2f} ({tn}/{tn+fp})")

## What Changed (2D Pipeline)

| Old | New |
|---|---|
| Hardcoded file | Multi-patient chb01+chb02 |
| Hardcoded timestamps | parse_seizure_summary() |
| Filter per-window | Filter ONCE before windowing |
| overlap=0.5 | **overlap=0.3** |
| fc1=Linear(16*64*3,32) | **AdaptiveAvgPool2d** |
| train_test_split | **Split by patient** |

---
# 1D CNN Pipeline -- All 23 Channels, Raw Time-Domain, With Augmentation

**Why:** Conv1d slides along time directly. All 23 channels capture spatial
diversity. Conservative augmentation on seizure windows only.

Run Cells A1-A9 below. The 2D pipeline above (Cells 1-22) still works independently.

## Cell A1: Process Training Set (Raw Windows)

`return_windows=True` -> shape `(N, 23, 1792)`

In [ ]:
print("\n=== TRAIN SET (RAW, 23 CHANNELS) ===")
X_train_1d, y_train_1d = collect_recordings(
    TRAIN_PATIENTS, SEIZURE_MAP, BASE_DIR,
    window_size_sec=WINDOW_SIZE_SEC, overlap=OVERLAP,
    return_windows=True)

print(f"\nTrain (1D): X={X_train_1d.shape}, y={y_train_1d.shape}")
print(f"Seizure: {np.sum(y_train_1d)}, Non-seizure: {len(y_train_1d)-np.sum(y_train_1d)}")

## Cell A2: Process Test Set (Raw Windows)

In [ ]:
print("\n=== TEST SET (RAW, 23 CHANNELS) ===")
X_test_1d, y_test_1d = collect_recordings(
    TEST_PATIENTS, SEIZURE_MAP, BASE_DIR,
    window_size_sec=WINDOW_SIZE_SEC, overlap=OVERLAP,
    return_windows=True)

print(f"\nTest (1D): X={X_test_1d.shape}, y={y_test_1d.shape}")
print(f"Seizure: {np.sum(y_test_1d)}, Non-seizure: {len(y_test_1d)-np.sum(y_test_1d)}")

## Cell A3: Balance

In [ ]:
def bal(data, labels, n_non=200, seed=42):
    np.random.seed(seed)
    si=np.where(labels==1)[0]; ni=np.where(labels==0)[0]
    cn=np.random.choice(ni,size=min(n_non,len(ni)),replace=False)
    k=np.concatenate([si,cn]); np.random.shuffle(k)
    return data[k],labels[k]

X_train_1d,y_train_1d = bal(X_train_1d,y_train_1d,NON_SEIZURE_SAMPLES,RANDOM_SEED)
X_test_1d,y_test_1d   = bal(X_test_1d,y_test_1d,NON_SEIZURE_SAMPLES,RANDOM_SEED)
ut,ct=np.unique(y_train_1d,return_counts=True); print(f"Train: {X_train_1d.shape} {dict(zip(ut,ct))}")
ut,ct=np.unique(y_test_1d,return_counts=True);  print(f"Test:  {X_test_1d.shape} {dict(zip(ut,ct))}")

## Cell A4: Normalize (Per-Channel)

Each EEG channel normalized independently. Stats from TRAIN only.

In [ ]:
X_train_1d=X_train_1d.astype(np.float32); X_test_1d=X_test_1d.astype(np.float32)
tm1=X_train_1d.mean(axis=(0,2),keepdims=True)
ts1=X_train_1d.std(axis=(0,2),keepdims=True)
X_train_1d=(X_train_1d-tm1)/(ts1+1e-8); X_test_1d=(X_test_1d-tm1)/(ts1+1e-8)
print(f"Normalized per-channel. Shape: {X_train_1d.shape}")

## Cell A5: Data Augmentation (Conservative, Seizure Only)

1. Time shift +-200ms
2. Amplitude x0.85-1.15
3. Gaussian noise sigma=0.01

Applied ONLY to seizure windows during training.

In [ ]:
def augment_seizure(windows, seed=None):
    """Augment seizure windows with time shift + amplitude + noise."""
    if seed is not None: np.random.seed(seed)
    aug = windows.copy()
    B,C,T = aug.shape
    for b in range(B):
        shift = np.random.randint(-51,51)
        aug[b] = np.roll(aug[b], shift, axis=-1)
        aug[b] *= np.random.uniform(0.85,1.15)
        aug[b] += np.random.randn(C,T).astype(np.float32)*0.01
    return aug
print("Augmentation ready.")

## Cell A6: Tensors + DataLoader

In [ ]:
Xtr1=torch.tensor(X_train_1d,dtype=torch.float32); ytr1=torch.tensor(y_train_1d,dtype=torch.long)
Xte1=torch.tensor(X_test_1d,dtype=torch.float32); yte1=torch.tensor(y_test_1d,dtype=torch.long)
tds1=TensorDataset(Xtr1,ytr1); eds1=TensorDataset(Xte1,yte1)
tl1=DataLoader(tds1,batch_size=BATCH_SIZE,shuffle=True)
el1=DataLoader(eds1,batch_size=BATCH_SIZE,shuffle=False)
print(f"Input: {Xtr1.shape} (batch,23,1792)")
print(f"Train batches: {len(tl1)} | Test: {len(el1)}")

## Cell A7: EEGCNN1D

3 Conv1d blocks (k7,k5,k3) + BatchNorm + Dropout + AdaptiveAvgPool1d(16)

In [ ]:
class EEGCNN1D(nn.Module):
    def __init__(self, nc=23):
        super().__init__()
        self.conv1=nn.Conv1d(nc,32,7,padding=3); self.bn1=nn.BatchNorm1d(32); self.pool1=nn.MaxPool1d(4)
        self.conv2=nn.Conv1d(32,64,5,padding=2); self.bn2=nn.BatchNorm1d(64); self.pool2=nn.MaxPool1d(4)
        self.conv3=nn.Conv1d(64,128,3,padding=1); self.bn3=nn.BatchNorm1d(128)
        self.adapt=nn.AdaptiveAvgPool1d(16)
        self.drop1=nn.Dropout(0.4); self.fc1=nn.Linear(128*16,64)
        self.drop2=nn.Dropout(0.3); self.fc2=nn.Linear(64,2)
    def forward(self,x):
        x=self.pool1(F.relu(self.bn1(self.conv1(x))))
        x=self.pool2(F.relu(self.bn2(self.conv2(x))))
        x=F.relu(self.bn3(self.conv3(x)))
        x=self.adapt(x); x=torch.flatten(x,1)
        x=self.drop1(x); x=F.relu(self.fc1(x))
        x=self.drop2(x); x=self.fc2(x)
        return x

model_1d = EEGCNN1D()
d=torch.randn(1,23,X_train_1d.shape[2])
with torch.no_grad(): o=model_1d(d)
print(f"Model: {sum(p.numel() for p in model_1d.parameters()):,} params")
print(f"Input {tuple(d.shape)} -> Output {tuple(o.shape)}  OK!")

## Cell A8: Train 1D CNN with Augmentation

1 augmented copy per seizure window per epoch.

In [ ]:
crit1=nn.CrossEntropyLoss(); opt1=optim.Adam(model_1d.parameters(),lr=LEARNING_RATE)
for ep in range(NUM_EPOCHS):
    model_1d.train(); rl=0.0; nb=0
    for x,y in tl1:
        sm=(y==1)
        if sm.any():
            si=x[sm]; sl=y[sm]
            an=augment_seizure(si.numpy())
            at=torch.tensor(an,dtype=torch.float32)
            xa=torch.cat([x,at],0); ya=torch.cat([y,sl],0)
        else: xa=x; ya=y
        opt1.zero_grad()
        o=model_1d(xa); l=crit1(o,ya)
        l.backward(); opt1.step()
        rl+=l.item(); nb+=1
    print(f"Epoch {ep+1}/{NUM_EPOCHS}, Loss: {rl/nb:.4f}")

## Cell A9: Evaluate 1D CNN on chb03

In [ ]:
model_1d.eval(); c,t=0,0; ap,at=[],[]
with torch.no_grad():
    for x,y in el1:
        o=model_1d(x); _,p=torch.max(o,1)
        ap.extend(p.cpu().numpy()); at.extend(y.cpu().numpy())
        t+=y.size(0); c+=(p==y).sum().item()
ap=np.array(ap); at=np.array(at)
print(f"Accuracy: {100*c/t:.2f}% ({c}/{t})")
sm=at==1
if sm.sum()>0:
    tp=(ap[sm]==1).sum(); fn=(ap[sm]==0).sum()
    print(f"Sensitivity: {tp/(tp+fn):.2f} ({tp}/{tp+fn})")
nm=at==0
if nm.sum()>0:
    tn=(ap[nm]==0).sum(); fp=(ap[nm]==1).sum()
    print(f"Specificity: {tn/(tn+fp):.2f} ({tn}/{tn+fp})")